<a href="https://colab.research.google.com/github/sensiwise-ai/agentic-ai-workshop2026-labs/blob/main/Lab2_SQL_Data_Analyst_Day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 · Data Analyst Pipeline: plain English → SQL → answer

Most of a company's knowledge is not in documents but in **tables**: sales, customers, invoices, tickets. Lab 1 gave the model documents to read; here we give it a **database to query**. You will build a text-to-SQL pipeline whose every step is visible, on a real sales database with 11 tables, then point the same pipeline at your own spreadsheet.

This lab focuses on constructing a robust pipeline that translates natural language questions into executable SQL queries, retrieves data, and then explains the results. The process involves several key steps, each designed to overcome the limitations of large language models (LLMs) by giving them structured access to data:

1.  **Catalogue the tables**: Understand the database schema. This step is crucial because the LLM needs to know what data is available and how it's structured before it can formulate a query.
2.  **Choose relevant tables**: Select only the necessary tables for a given question. This optimizes token usage and prevents the LLM from being overwhelmed with irrelevant schema information.
3.  **Inject their schema**: Provide detailed schema information for the chosen tables. This helps the LLM understand column names, data types, and relationships, which are essential for writing correct SQL.
4.  **Write SQL**: Generate the SQL query based on the question and schema. This is where the LLM translates the natural language request into a database-executable command.
5.  **Validate + run**: Ensure the SQL is safe and execute it. Validation is critical for security (preventing malicious queries) and correctness, while execution retrieves the actual data.
6.  **Error handling (retry)**: If the SQL fails, feed the error back to the model and retry. This allows the LLM to learn from its mistakes and self-correct, improving its SQL generation over time.
7.  **Explain the result**: Summarize the query results in plain English. This translates the raw data into actionable insights for the end-user, making the pipeline valuable for non-technical stakeholders.

Libraries do the plumbing: **LangChain's `SQLDatabase`** inspects tables and formats schemas (to provide structured information to the LLM), **`sqlglot`** parses the generated SQL (for validation and safety), **pandas** runs it and shows results (for data retrieval and display). The model makes exactly three decisions: which tables (to focus its attention), which SQL (to fetch the data), what the numbers mean (to provide a human-readable answer).

### Setup: Runtime → Change runtime type → **T4 GPU**, then run this cell

To ensure optimal performance for the large language model (LLM), it is crucial to switch the Colab runtime to a **T4 GPU**. This provides the necessary computational resources for accelerating model inference, which is a computationally intensive task. Once the runtime is set, execute this cell to install the required Python packages and load the model, preparing the environment for the pipeline's operations.

In [ ]:
# Install necessary libraries:
# `transformers` for integrating with large language models,
# `accelerate` for optimizing model performance,
# `langchain-community` for SQL database utilities and schema extraction,
# `sqlglot` for robust SQL parsing and validation, and
# `openpyxl` for handling Excel file uploads.
!pip install -q -U transformers accelerate langchain-community sqlglot openpyxl

# Import core Python modules.
import torch  # PyTorch library for deep learning operations
import json   # For working with JSON data
import re     # For regular expressions

# Import specific classes from the transformers library.
from transformers import AutoTokenizer, AutoModelForImageTextToText # AutoTokenizer loads the correct tokenizer for a model, AutoModelForImageTextToText loads the model itself.


### Loading the Language Model (LLM)

In [ ]:
# Define the ID of the pre-trained language model to be used.
# 'Qwen/Qwen3.5-4B' is a 4 billion parameter open model, approximately 9 GB in FP16 precision.
# It's suitable for a free T4 GPU instance in Colab.
MODEL_ID = "Qwen/Qwen3.5-4B"          # 4B open model, ~9 GB in fp16, fits a free T4. (Fallback: "Qwen/Qwen3-4B")

# Load the tokenizer associated with the specified model ID.
# The tokenizer is responsible for converting text into numerical tokens that the model can understand.
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the pre-trained language model itself.
# `dtype=torch.float16` uses half-precision floating points for reduced memory usage and faster computation.
# `device_map="auto"` automatically distributes the model across available devices (e.g., CPU and GPU).
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

### Defining the `chat` function for interacting with the LLM

In [ ]:
def chat(messages, max_new_tokens=400, temperature=0.0):
    """The one function that talks to the model: a prompt string or a list of {role, content} -> reply text."""
    # If the input `messages` is a single string, convert it into the expected list format.
    # The model expects a list of dictionaries, where each dictionary has a 'role' (e.g., 'user') and 'content'.
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    # Apply the model's specific chat template to format the messages into a single prompt string.
    # `tokenize=False` means it returns a string, not token IDs.
    # `add_generation_prompt=True` adds a token to signal the model to start generating.
    # `enable_thinking=False` (specific to some models) disables internal 'thinking' steps.
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)  # wrap in the model's own chat markup

    # Convert the formatted prompt string into token IDs and move them to the model's device (e.g., GPU).
    # `return_tensors="pt"` ensures the output is a PyTorch tensor.
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    # Generate a response from the model.
    # `**inputs` unpacks the tokenized input.
    # `max_new_tokens` limits the length of the generated response.
    # `temperature=0.0` makes the output deterministic (less creative, more factual).
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decode the generated token IDs back into a human-readable string.
    # `out[0][inputs["input_ids"].shape[1]:]` slices the output to remove the input prompt tokens, keeping only the generated part.
    # `skip_special_tokens=True` removes special tokens like `<s>`, `</s>`, etc.
    # `.strip()` removes leading/trailing whitespace.
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # drop the prompt, keep only what was generated

## 0 · The problem: the model has no hands

Without direct access to external tools or databases, a large language model (LLM) is limited to its pre-trained knowledge. It cannot execute queries, retrieve real-time data, or interact with external systems. This section demonstrates this inherent limitation, setting the stage for building a pipeline that gives the model 'hands' to interact with a database, showing *why* such a pipeline is necessary.

In [ ]:
# This demonstrates the model's inability to access external, real-time data or perform database queries directly,
# highlighting its inherent limitation without external tools.
print(chat("In our company's sales database, which three countries bring in the most revenue?"))

## 1 · The data: a real sales database

This lab uses the [Chinook database](https://github.com/lerocha/chinook-database), which simulates a digital music store. It comprises tables for customers, invoices, invoice lines, tracks, albums, artists, genres, and employees—a total of eleven interconnected tables in a 1 MB SQLite file. This structure is representative of a small company's ERP or CRM export, making it an ideal real-world example to demonstrate the text-to-SQL pipeline.

The `connect()` function is central to pointing the pipeline at any SQLite file. It performs three key actions: establishing a connection for `pandas` to read data (so we can execute SQL and retrieve results), initializing an inspector for `LangChain` to understand the database schema (to provide structured schema information to the LLM), and generating a compact catalogue of tables (for efficient initial table selection by the LLM). The same function will be reused later to connect to your own spreadsheet data, emphasizing its reusability.

In [ ]:
import urllib.request, sqlite3, pandas as pd # Import necessary libraries: urllib for downloading files from the internet, sqlite3 for connecting to and interacting with SQLite databases, and pandas for data manipulation and SQL execution.
from IPython.display import display # Import display for better output formatting in Colab, allowing for rich display of DataFrames.
from langchain_community.utilities import SQLDatabase # Import SQLDatabase from langchain_community to interact with SQL databases and extract schema information in a structured way for the LLM.

def step(n, title, *lines):
    '''One banner per pipeline step, so every run reads the same way.'''
    # Print a formatted banner for each step of the pipeline. This helps visualize the flow of execution and debug the pipeline.
    print(f"\n{'━' * 72}\n Step {n} · {title}\n{'━' * 72}")
    for line in lines:
        print(line)

def connect(path):
    '''Point the pipeline at a SQLite file -> (connection for pandas, inspector for LangChain, compact catalogue).'''
    # Establish a connection to the SQLite database. This connection is used by pandas to execute SQL queries.
    conn = sqlite3.connect(path)
    # Create a SQLDatabase instance for LangChain, including sample rows for detailed schema info. This provides the LLM with concrete examples of data values, aiding in accurate SQL generation.
    db = SQLDatabase.from_uri(f"sqlite:///{path}", sample_rows_in_table_info=3)
    # Create another SQLDatabase instance for a compact catalogue (no sample rows), for overview. This lightweight catalogue is used for initial table selection by the LLM, reducing token cost.
    catalogue = SQLDatabase.from_uri(f"sqlite:///{path}", sample_rows_in_table_info=0).table_info
    # Log the number of tables and the catalogue size, and list usable table names. This provides an overview of the database structure to the user.
    step(1, "Catalogue", f"{len(db.get_usable_table_names())} tables, {len(catalogue):,} characters: {db.get_usable_table_names()}")
    return conn, db, catalogue # Return the connection (for pandas), LangChain's DB object (for detailed schema), and the compact catalogue (for table selection).

# Download the Chinook SQLite database file from GitHub. This fetches the sample database to work with.
urllib.request.urlretrieve("https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite", "chinook.db")
# Connect to the downloaded database and store the connection, db object, and catalogue. This sets up the database for the pipeline.
conn, db, catalogue = connect("chinook.db")
# Read the first 5 rows from the 'Invoice' table into a pandas DataFrame and display it. This provides a quick visual check of the data.
pd.read_sql("SELECT * FROM Invoice LIMIT 5", conn)

## Step 1 · Catalogue the tables

In this initial step, the `SQLDatabase` utility reads the database's own metadata to generate a comprehensive description of its structure. This catalogue includes details for every table, such as their `CREATE TABLE` statements, column names, data types, primary keys, and crucially, **foreign keys**. The presence of foreign keys is vital, as they enable the model to correctly infer relationships between tables and construct accurate `JOIN` operations, *which is why* this detailed schema information is crucial for the LLM to write correct SQL. The catalogue is built only once during the `connect()` call, ensuring efficiency for subsequent queries. This initial overview helps the model understand the available data entities and their interconnections.

In [ ]:
print(catalogue[:2000], "...") # Print the first 2000 characters of the catalogue to show the schema information. This allows us to inspect the raw schema data that the LLM will use for table selection.

## Step 2: Choose the tables for the question, the model's first decision

Providing the entire database schema with sample rows for every table to the large language model (LLM) would be highly inefficient, potentially consuming thousands of tokens per question, most of which would be irrelevant. To optimize this, the model's first critical decision is to identify and select only the tables directly relevant to answering the user's question. It does this by reading the compact catalogue (table names, columns, keys) and then naming the required tables, along with a justification. The pipeline then filters this selection, retaining only the table names that actually exist in the database, ensuring valid references for the subsequent steps. This proactive selection *is why* this step significantly reduces the token cost and improves the LLM's ability to focus on pertinent information.

In [ ]:
def chat_json(messages, **kw):
    """Same as chat(), but returns the first JSON object found in the reply as a dict."""
    # Call the main chat function to get a text response from the model. This leverages the base chat functionality to get the LLM's raw output.
    text = chat(messages, **kw)
    # Use a regular expression to find the first JSON object in the text. This is necessary because the LLM might embed JSON within other explanatory text.
    match = re.search(r"\{.*\}", text, re.S)
    # If a JSON object is found, parse it and return as a dictionary; otherwise, return an empty dictionary. This ensures a structured output for programmatic use.
    return json.loads(match.group()) if match else {}

In [ ]:
def choose_tables(question):
    '''The model reads the catalogue (names, columns, keys) and picks the tables this question needs -> (tables, why).'''
    # Construct the prompt for the model, including the database catalogue and the user's question. This provides the LLM with all necessary context to make a decision.
    # Instruct the model to return its answer as a JSON object with 'tables' and 'why' fields. This forces a structured output that can be easily parsed.
    reply = chat_json(f"Database catalogue:\n{catalogue}\n\nQuestion: {question}\n\n"
                      f'Which tables are needed to answer it? Usually one to three, plus any table needed to join them. '
                      f'Return JSON: {{"tables": [...], "why": "..."}}')
    # Extract table names from the model's reply and filter them to ensure they exist in the database. This validates the LLM's choices against the actual database schema.
    # If no tables are returned or valid, default to all usable tables to prevent pipeline failure. This acts as a fallback mechanism.
    tables = [t for t in reply.get("tables", []) if t in db.get_usable_table_names()] or list(db.get_usable_table_names())
    return tables, reply.get("why", "") # Return the chosen tables and the model's explanation, providing transparency into the LLM's reasoning.

# Define the question for which tables need to be chosen. This is the user's input we are trying to answer.
QUESTION = "Which five countries bring in the most revenue, and how much each?"
# Call choose_tables to get the relevant tables and the reason for their selection. This executes the model's first decision-making step.
tables, why = choose_tables(QUESTION)
# Log the chosen tables and the model's justification using the step function. This makes the LLM's decision visible and traceable.

## Step 3 · Inject the schema of just those tables

After the model has identified the most relevant tables for the given question, the next crucial step is to provide a detailed schema for only those selected tables. Instead of simply listing column names, this process injects the full `CREATE TABLE` statement for each chosen table, **along with three sample rows of actual data**. This is significantly more informative than just column names because it allows the model to observe real values, understand data formats (e.g., date formats, country spellings, currency scales), and infer potential data characteristics that might influence SQL query generation. By seeing concrete examples, the model can avoid common errors related to data types or value representations, *which is why* this rich context is provided to the LLM before it attempts to write SQL.

In [ ]:
# Retrieve the detailed schema information for the chosen tables, including sample rows. This function from LangChain's SQLDatabase is designed to extract comprehensive schema details.
schema = db.get_table_info(tables)
# Log the number of tables, the schema size, and indicate that CREATE TABLE statements with sample rows are included. This provides an audit trail of the information being fed to the LLM.
step(3, "Inject schema", f"{len(tables)} tables, {len(schema):,} characters: CREATE TABLE + 3 sample rows each")
print(schema) # Print the generated schema to inspect its content. This allows a human to verify the schema context.

## Step 4 · Write the SQL — the model's second decision

This is where the large language model (LLM) makes its second critical decision: generating the actual SQL query. The prompt provided to the model includes the detailed schema (from Step 3), the original natural language question, and a set of explicit rules for SQL generation. These rules ensure that the generated SQL adheres to best practices, such as using only existing tables and columns, correctly joining tables via foreign keys, and applying `GROUP BY` clauses appropriately for aggregate functions. The model is instructed to encapsulate its SQL output within a ````sql` block. The pipeline then extracts this SQL block using a regular expression. Importantly, at this stage, no SQL is executed; it is merely generated. If a query fails in a later step, the error message and the failed query will be fed back into this prompt for a corrected attempt, *which is why* capturing the SQL output and allowing for retries is fundamental to this pipeline.

In [ ]:
def write_sql(question, schema, error=None, failed_sql=None, temperature=0.0):
    # Construct the prompt for the model to generate SQL, including dialect, schema, question, and rules. This provides the LLM with all necessary information and constraints for SQL generation.
    prompt = (f"You write {db.dialect} SQL.\n\nSchema:\n{schema}\n\nQuestion: {question}\n\n"
              f"Rules: one SELECT statement; use only tables and columns that appear in the schema, spelled exactly; "
              f"join through the foreign keys; if you use an aggregate, every column you select or order by that is not "
              f"itself aggregated must appear in GROUP BY; "
              f"LIMIT 20 unless the question asks for a count. Reply with the SQL in a ```sql block.")
    # If there was a previous error, append it to the prompt to guide correction. This allows the LLM to learn from past mistakes and refine its SQL query.
    if error:
        prompt += f"\n\nYour previous query:\n{failed_sql}\n\nfailed with this error:\n{error}\n\nWrite a corrected query."
    # Call the chat function to get the model's SQL response. This is where the LLM generates the SQL.
    reply = chat(prompt, max_new_tokens=300, temperature=temperature)
    # Use a regular expression to extract the SQL code from within the ```sql block. This ensures that only the SQL code is extracted, separating it from any conversational text.
    block = re.search(r"```sql\s*(.*?)```", reply, re.S | re.I)
    # Return the extracted SQL or the raw reply if no block is found, stripped of leading/trailing whitespace and semicolons. This provides a clean SQL string for execution.
    return (block.group(1) if block else reply).strip().rstrip(";")

# Call write_sql to generate the initial SQL query for the defined QUESTION. This initiates the SQL generation process.
sql = write_sql(QUESTION, schema)
# Log the generated SQL using the step function. This makes the generated SQL visible to the user for review.

## Step 5 · Validate, then run

Before executing any generated SQL query, two critical checks are performed to ensure both safety and correctness. First, the `sqlglot` library is used to parse the SQL. This parsing step rigorously verifies that the query is indeed a **SELECT** statement, preventing potentially destructive commands like `DROP TABLE` from ever reaching the database. This acts as a crucial security measure and *is why* we parse the SQL before execution. Second, if the query passes validation, `pandas` executes it against the database, returning the results as a DataFrame. If the database itself rejects the query due to syntax errors or other issues, the specific error message is captured. This message is vital for Step 6, where it will be used to inform the model for a retry, *which is why* we wrap the execution in a try-except block.

In [ ]:
import sqlglot # Import sqlglot for SQL parsing and validation. This library helps ensure the generated SQL is syntactically correct and safe.

def run_sql(sql):
    '''Parse with sqlglot (must be a SELECT), then execute with pandas.'''
    # Parse the SQL query using sqlglot, specifying the database dialect. This checks for syntax errors and validates the query structure.
    tree = sqlglot.parse_one(sql, read=db.dialect)
    # Check if the parsed statement is a SELECT or UNION; if not, raise an error for safety. This prevents execution of potentially harmful non-SELECT statements like DROP TABLE.
    if not isinstance(tree, (sqlglot.exp.Select, sqlglot.exp.Union)):
        raise ValueError("only SELECT statements are allowed")
    # Execute the SQL query using pandas and the established database connection. Pandas is used here for its convenience in running SQL and returning DataFrames.
    return pd.read_sql(sql, conn)

def db_error(e):
    '''The reason, not the echo: pandas prepends the whole query to the database's message, the diagnosis is the last line.'''
    # Extract the actual database error message from the exception, typically the last line. This is done to get a concise and relevant error message to feed back to the LLM.
    return str(e.__cause__ or e).strip().splitlines()[-1]

try:
    # Attempt to run the generated SQL query. This is the core execution of the SQL.
    result = run_sql(sql)
    # If successful, log the validation status and the dimensions of the result DataFrame, then display the result. This confirms successful execution and shows the retrieved data.
    step(5, "Validate + run", "SELECT only ✓", f"{len(result)} rows × {len(result.columns)} columns"); display(result)
except Exception as e:
    # If an exception occurs during SQL execution, capture and format the database error. This handles potential issues during query execution.
    error = db_error(e)
    # Log the retry step with the database error message, indicating a failure to be handled in step 6. This communicates the error to the user and prepares for LLM feedback.

## Step 6 · When the SQL fails, feed the error back and try again

It is common for generated SQL queries to fail initially, whether due to a misspelt column, an incorrect join condition, or the use of a function not supported by the specific database dialect. When such a failure occurs, the database provides a precise error message. This message is then ingeniously fed back into the prompt for the model, along with the original failed query. This iterative feedback loop allows the model to learn from its mistakes and generate a corrected query. This section first demonstrates a deliberate error to illustrate the repair mechanism and then introduces the automated loop that performs these retries. The first attempt is typically `greedy` (temperature 0) to get a direct answer, while subsequent retries introduce a small amount of `randomness` (temperature > 0) to encourage the model to explore alternative solutions and avoid getting stuck in repetitive errors. This self-correction mechanism *is why* the pipeline is robust and can handle initial LLM errors.

In [ ]:
broken = '''SELECT BillingCountry, SUM(Totals) AS revenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY revenue DESC LIMIT 5''' # Define a syntactically incorrect SQL query to demonstrate error handling; 'Totals' is intentionally wrong to trigger an error.
try:
    run_sql(broken) # Attempt to run the broken SQL query. This is expected to fail.
except Exception as e:
    error = db_error(e) # Capture the specific database error message. This extracts the exact reason for failure from the database.
    # Log the retry step, showing the error and explaining that it will be fed back to the model. This informs the user about the error and the next step.
    step(6, "Retry", f"database says: {error}", "→ the failed query and this message go back into the prompt")
    # Call write_sql again with the original question, schema, the error, and the failed SQL to generate a corrected query. This demonstrates how the LLM uses error feedback to self-correct its SQL.
    step(4, "Write SQL (model) · attempt 2", write_sql(QUESTION, schema, error=error, failed_sql=broken))

In [ ]:
def query(question, schema, max_attempts=3):
    '''Steps 4-6: write SQL, validate + run, feed the error back, until it works. Returns (sql, result) or (sql, None).'''
    sql, error = None, None # Initialize sql and error variables to None before starting attempts.
    for attempt in range(1, max_attempts + 1):
        # Write SQL, varying the temperature for retries: greedy (0.0) for the first attempt, then some randomness (0.3). This strategy balances directness with exploration in subsequent attempts.
        sql = write_sql(question, schema, error, failed_sql=sql, temperature=0.0 if attempt == 1 else 0.3)
        # Log the current attempt to write SQL. This provides visibility into the retry process.
        step(4, f"Write SQL (model) · attempt {attempt}", sql or "(the model returned no SQL)")
        try:
            result = run_sql(sql) # Attempt to run the generated SQL query. This tests the validity and executability of the SQL.
            # If successful, log validation and result dimensions, display the head of the result, and return sql and result. This indicates a successful query and shows the initial data.
            step(5, "Validate + run", "SELECT only ✓", f"{len(result)} rows × {len(result.columns)} columns"); display(result.head(20))
            return sql, result
        except Exception as e:
            error = db_error(e) # Capture the database error message. This extracts the specific error for feedback.
            # Log the retry step, indicating the database error and the feedback mechanism. This informs the user about the failure and how the LLM will attempt to fix it.
            step(6, "Retry", f"database says: {error}", "→ the failed query and this message go back into the prompt")
    return sql, None # If all attempts fail, return the last generated SQL and None for result. This indicates that the query could not be resolved automatically.

# Call the query function with the QUESTION and schema to get the SQL and result. This initiates the iterative SQL generation and execution process.
sql, result = query(QUESTION, schema)

## Step 7 · Explain the result in plain English — the model's third decision

A raw table of numbers is often not sufficient as a final answer for a business user. In this concluding step, the large language model (LLM) makes its third and final decision: to articulate the meaning of the query result in plain English. The model is provided with the original question, the executed SQL query, and the resulting data table. Its task is to synthesize this information into a concise, manager-friendly explanation, typically two or three sentences long. A crucial constraint is that the explanation must exclusively use numerical values that are present within the result table, ensuring factual accuracy and preventing hallucination. This step bridges the gap between technical data retrieval and actionable business insights, *which is why* it's vital for delivering value to the end-user.

In [ ]:
def explain(question, sql, result):
    # Construct the prompt for the model to explain the result, including the question, SQL, and result table. This gives the LLM all the context needed to summarize the findings.
    # Instruct the model to answer in two or three sentences for a manager, using only numbers from the table. This enforces conciseness, relevance, and factual accuracy, preventing hallucination.
    return chat(f"Question: {question}\nSQL used: {sql}\nResult table:\n{result.head(20).to_string(index=False)}\n\n"
                f"Answer the question in two or three sentences for a manager. Use only numbers that appear in the table.", max_new_tokens=200)

# Log the explanation step. If a result exists, call explain to generate the natural language summary; otherwise, indicate no result to explain. This makes the final answer visible or reports a failure to explain.
step(7, "Explain (model)", explain(QUESTION, sql, result) if result is not None else "no result to explain - the query never worked")

## The whole pipeline in one function

This section consolidates all the individual steps (1 to 7) of the text-to-SQL pipeline into a single, cohesive function named `ask()`. The `ask()` function orchestrates the entire process: it leverages the catalogue from `connect()` (Step 1, for initial database overview), then calls `choose_tables` (Step 2, for model-driven table selection), followed by `get_table_info` (Step 3, to inject the detailed schema for selected tables). The core execution and retry logic are handled by `query` (Steps 4-6, for SQL generation, validation, and self-correction), and finally, `explain` (Step 7, to provide a natural language summary). In this structured approach, the code dictates the fixed order of operations, while the model is responsible for making precisely three decisions: selecting tables, generating SQL, and explaining the results. Each step prominently prints a banner detailing its decision, which is invaluable for debugging, as any incorrect answer can be traced back to the specific step that introduced the error. This unified function *is why* the pipeline is easy to use and provides clear transparency.

In [ ]:
def ask(question):
    # Print the user's question to clearly state what is being asked.
    print(f"\nQUESTION: {question}")
    # Step 1: Catalogue tables (already built in connect() and used here for logging). This shows the available tables in the database.
    step(1, "Catalogue", f"{len(db.get_usable_table_names())} tables available (built once in connect())")
    # Step 2: Choose relevant tables for the question using the model. This is the LLM's first decision to narrow down the scope.
    tables, why = choose_tables(question);      step(2, "Choose tables (model)", f"tables: {tables}", f"why: {why}")
    # Step 3: Inject the detailed schema for the chosen tables. This provides the necessary context for the LLM to write accurate SQL.
    schema = db.get_table_info(tables);         step(3, "Inject schema", f"{len(tables)} tables, {len(schema):,} characters: CREATE TABLE + 3 sample rows each")
    # Steps 4-6: Generate, validate, run SQL, and retry if necessary. These steps handle the core SQL interaction and error correction.
    sql, result = query(question, schema)       # steps 4-6 print themselves for transparency
    # If no working query is found after retries, log a message and return. This informs the user if the pipeline could not resolve the query automatically.
    if result is None:
        step(7, "Explain (model)", "No working query after three attempts - hand to a human."); return
    # Step 7: Explain the query result in plain English using the model. This translates the data into an understandable answer for the user.
    step(7, "Explain (model)", explain(question, sql, result))

# Example usage: Ask the pipeline a question about genres. This demonstrates the end-to-end functionality of the 'ask' function.
ask("Which three genres sell the most tracks?")

In [ ]:
# Example usage: Ask the pipeline a question that requires joining multiple tables (Employee, Customer, Invoice). This showcases the pipeline's ability to handle complex queries involving multiple data sources.
ask("Which employee looks after the customers who spent the most in total?")
# Example usage: Ask the pipeline a question that involves date functions for aggregation. This demonstrates the pipeline's capability to process temporal queries.
ask("How did total revenue change from year to year?")

## The same pipeline in LangChain

LangChain, a popular framework for developing LLM applications, encapsulates much of the text-to-SQL pipeline into a single, convenient function: `create_sql_query_chain`. This function inherently handles several steps: it reads the database catalogue (akin to our Step 1), injects the relevant schema with sample rows (similar to our Step 3), intelligently prompts the model, and ultimately returns the generated SQL. To integrate our custom local model into LangChain, a small adapter class (`LocalModel`) is required. This adapter effectively wraps our `chat()` function, allowing LangChain to call it when model inference is needed. By inspecting the hidden prompt generated by LangChain's chain, we can observe how steps 1-4 are integrated. We then execute LangChain's generated SQL through our `run_sql` function for consistent validation and execution, enabling a direct comparison with the SQL produced by our custom pipeline. This section *is why* we can compare the efficiency and transparency of our custom pipeline with a popular framework solution.

In [ ]:
from langchain_core.language_models.llms import LLM # Import the base LLM class from langchain_core, which is necessary for creating a custom LLM wrapper for LangChain.
try:
    from langchain_classic.chains import create_sql_query_chain            # Attempt to import from langchain_classic for newer versions of LangChain.
except ImportError:
    from langchain.chains import create_sql_query_chain                    # Fallback to langchain.chains for older versions, ensuring compatibility.

class LocalModel(LLM):
    """The adapter: LangChain calls _call(prompt), we call chat(). Nothing else is needed."""
    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        text = chat(prompt, max_new_tokens=300) # Call our custom chat function with the prompt. This integrates our LLM into the LangChain ecosystem.
        for s in stop or []:
            text = text.split(s)[0]                                         # Honour any stop sequences requested by LangChain. This ensures the generated text respects LangChain's stopping criteria.
        return text
    @property
    def _llm_type(self):
        return "qwen-local" # Define the type of this local LLM. This is a property required by LangChain for identification.

# Instantiate our LocalModel. This creates an instance of our custom LLM wrapper.
llm = LocalModel()
# Create an SQL query chain using the local LLM and the SQLDatabase object. This leverages LangChain's pre-built logic for text-to-SQL, using our LLM.
chain = create_sql_query_chain(llm, db)                                    # catalogue -> schema -> prompt -> model -> SQL, packaged by LangChain.
# Print the template of the prompt used by LangChain's SQL query chain (first 900 characters). This reveals the underlying prompt structure LangChain uses, which hides some of the steps from our custom pipeline.

In [ ]:
raw = chain.invoke({"question": QUESTION}) # Invoke the LangChain SQL query chain with the predefined question. This executes LangChain's text-to-SQL process.
# Search for the SQL block within the raw output, either in ```sql format or as a SELECT statement. This extracts the generated SQL query from LangChain's response.
found = re.search(r"```sql\s*(.*?)```", raw, re.S | re.I) or re.search(r"(SELECT[\s\S]*?)(?=\n\s*\n|SQLResult|$)", raw, re.I)
# Extract and clean the generated SQL query. This prepares the SQL for execution by removing any surrounding text or formatting.
sql_langchain = (found.group(1) if found else raw).strip().rstrip(";")
print(sql_langchain) # Print the SQL query generated by LangChain for inspection.
display(run_sql(sql_langchain))                                             # Execute LangChain's SQL using our run_sql function and display the result. This allows for a direct comparison of results and execution between our custom pipeline and LangChain's.

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit # Import SQLDatabaseToolkit from langchain_community. This toolkit provides pre-built tools for interacting with SQL databases within a LangChain agent.

# Iterate through the tools provided by LangChain's SQL agent toolkit and print their names and descriptions. This helps understand the capabilities of LangChain's agent for SQL interactions.
for tool in SQLDatabaseToolkit(db=db, llm=llm).get_tools():               # These are the four tools LangChain's SQL agent is given to perform its operations.
    print(f"{tool.name:<24} {tool.description[:500]}")

Same steps, different surface. While LangChain's `create_sql_query_chain` effectively streamlines the text-to-SQL process, it abstracts away some of the intermediate steps, such as the explicit catalogue inspection and the choice of tables. This abstraction saves about thirty lines of code and benefits from a prompt specifically tuned over many databases. However, the custom pipeline built in this lab, with its visible step-by-step execution, offers greater transparency and debuggability. The toolkit's four tools (list tables, show schema, run query, check query) closely mirror our Steps 1, 3, 5, and 6. Integrating these tools into an agent loop, as explored in the Interactive lab, would effectively replicate LangChain's SQL agent functionality, highlighting that our custom pipeline provides a clear, understandable foundation for building more complex agentic systems. This comparison *is why* understanding both approaches is valuable.

**Pipeline, not agent.** The core distinction in this lab is the use of a fixed pipeline versus an autonomous agent. In our current setup, the steps and their sequence are predefined within the `ask()` function; the model's role is limited to making decisions within three specific steps (table selection, SQL generation, and explanation). This structured approach is ideal when the path to a solution is known in advance, offering benefits such as cost-efficiency, predictable behavior, and clear attribution of failures to specific steps. Lab 3 will delve into the alternative, where the model itself determines *whether* to take additional steps, introducing a more dynamic, agent-based paradigm. This distinction *is why* the current lab focuses on a fixed, transparent process.

## 🗂️ Your turn: your own spreadsheet

Now, it's your opportunity to apply this text-to-SQL pipeline to your own data. You can either upload an Excel workbook, where each sheet will be treated as a separate table, or provide multiple CSV files, each becoming its own table within a new SQLite database. The column names in your uploaded files will be automatically made SQL-friendly to ensure compatibility. Crucially, **the same `connect()` and the same `ask()` functions** that you've used with the Chinook database will now seamlessly operate on your personal data, providing a step-by-step analysis. All processing occurs securely within this Colab session, with no data leaving your environment. If you don't have a file readily available, you can use `sample_files/brightwave_sales.xlsx` from the workshop folder as an example. This section *is why* the lab provides a practical application for users to experiment with their own datasets.

In [ ]:
from google.colab import files # Import the files module to enable file uploads in Colab, allowing users to provide their own data.

def load_spreadsheets(uploads, path="my_data.db"):
    '''Every CSV, and every sheet of an Excel workbook, becomes one table in a new SQLite file.'''
    target = sqlite3.connect(path) # Create a new SQLite database file for the uploaded data. This provides a structured storage for the user's data.
    for name, blob in uploads.items():
        open(name, "wb").write(blob) # Write the uploaded file content to a local file. This makes the uploaded data accessible to the Python environment.
        # Determine if the file is CSV or Excel, then read its content into pandas DataFrames. This handles different common spreadsheet formats.
        sheets = {name.rsplit(".", 1)[0]: pd.read_csv(name)} if name.lower().endswith(".csv") else pd.read_excel(name, sheet_name=None)
        for sheet, frame in sheets.items():
            # Clean column names to be SQL-friendly (replace non-alphanumeric with underscores). This ensures that column names are valid for SQL queries, preventing errors.
            frame.columns = [re.sub(r"\W+", "_", str(c)).strip("_") for c in frame.columns]     # Example: "Amount (GBP)" -> "Amount_GBP"
            # Write each DataFrame (sheet or CSV) to a new table in the SQLite database. This populates the temporary database with the user's data.
            frame.to_sql(re.sub(r"\W+", "_", sheet), target, if_exists="replace", index=False)
    return path # Return the path to the newly created SQLite database. This path is used by the 'connect' function.

# Prompt the user to upload files, then load them into a new SQLite database. This initiates the file upload process.
# Connect to this new database and update the global conn, db, and catalogue objects for step 1. This reconfigures the pipeline to use the user's data.
conn, db, catalogue = connect(load_spreadsheets(files.upload()))      # step 1 on YOUR tables; ask() now uses them.

In [ ]:
while True:                                                                      # Loop indefinitely to allow multiple questions. This enables an interactive Q&A session with the data.
    q = input("Ask your data: ").strip() # Prompt the user to enter a question. This is where the user provides their natural language query.
    if q.lower() in ("quit", "exit", "q", ""): break # If the user types a quit command or empty, break the loop. This provides a way to exit the interactive session.
    ask(q) # Call the ask function to process the user's question through the pipeline. This executes the entire text-to-SQL pipeline for each user query.

If the SQL generated by the pipeline consistently fails on your uploaded file, it is essential to systematically review the output banners from each step. This structured debugging approach can quickly pinpoint the issue, *which is why* the pipeline provides such granular visibility:

*   **Step 2 (Choose tables)**: Did the model correctly identify the necessary tables for your question? If not, the table names or the question's phrasing might be unclear. This helps diagnose issues related to table selection.
*   **Step 3 (Inject schema)**: Are the column names and sample rows in the injected schema as expected after the cleaning process? Mismatches here often indicate an issue with how your data's column headers were processed. This helps identify problems with schema interpretation.
*   **Step 6 (Retry)**: What specific error message did the database return? This message is the most direct diagnostic clue. Common issues include `no such column`, `syntax error`, or `misuse of aggregate function`. This pinpoints the exact database error.

Experience shows that in nine out of ten cases, the solution lies in clarifying column names in your source data, providing more distinct sample rows, or adjusting the question's phrasing, rather than in fundamentally improving the underlying language model. The transparency of this pipeline makes such debugging straightforward.

## Exercises · Recap

To further solidify your understanding and extend the capabilities of this pipeline, consider the following exercises:

1.  **Self-check step**: Integrate an additional model-based validation between steps 4 and 5. Before running the SQL, prompt the model with: *"Does this SQL answer the question? Reply yes or no and why"*. Only proceed to execute the query if the model responds with 'yes'. This adds an extra layer of confidence before executing potentially costly or impactful queries.
2.  **Bigger catalogue**: Enhance the `choose_tables` function by providing a concise, one-line description for each table (these descriptions would be written manually). Compare how this enriched catalogue influences the model's choices for the "employee" question, potentially leading to more accurate or efficient table selections. This tests the impact of richer metadata on LLM decision-making.
3.  **Chart**: After successful execution in step 5, extend the pipeline to visualize the results. Use `run_sql(sql).plot(x=..., y=..., kind="bar")` to generate a chart, providing a graphical representation of the data. This adds a visual component to the data analysis, making results more accessible.
4.  **Towards an agent**: Introduce a feedback loop after step 7. Prompt the model with: *"Do you need another query to answer fully?"* If the model responds positively, loop back to an earlier step to generate a follow-up query. This mimics the foundational pattern of an autonomous agent, which will be explored in Lab 3. This pushes the boundaries of the pipeline towards more dynamic interaction.

| Step | Who decides | Function | Library |
|---|---|---|---|
| 1 Catalogue | code | `connect()` → `SQLDatabase(...).table_info` | LangChain |
| 2 Choose tables | **model** | `choose_tables()` | `chat_json` |
| 3 Inject schema | code | `db.get_table_info(tables)` | LangChain |
| 4 Write SQL | **model** | `write_sql()` | `chat` |
| 5 Validate + run | code | `run_sql()` | sqlglot, pandas |
| 6 Retry with error | code | loop in `query()` | — |
| 7 Explain | **model** | `explain()` | `chat` |

**Next → Interactive:** an agent that decides which tool to call, with a human approving the risky ones.